# Generate Dummy Data for Production Person IDs
Extracts unique person IDs from production invoice history and generates synthetic supplementary data for those persons.

Since `public.person`, `public.address`, and `scheduler.calendars_events` cannot be queried from production, this notebook generates realistic dummy equivalents keyed to the real production person IDs.

**Inputs:**
- `data/raw/invoice_cohorts/person_invoice_history_2025-04-20.csv` — production invoice history (~24K rows)
- `data/processed/census_zip_features_model_ready.csv` — valid US census zip codes

**Outputs:**
- `data/processed/prod_dummy_persons.csv`
- `data/processed/prod_dummy_addresses.csv`
- `data/processed/prod_dummy_calendar_events.csv`

## Step 1: Load Production Invoices & Extract Person IDs
Loads the full production invoice history and extracts all unique, non-null person IDs.

Computes each person's earliest invoice `createdat` — used as the baseline for `entry_date` generation in Step 2.

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import uuid
from datetime import timedelta
import pytz

np.random.seed(42)
fake = Faker()
Faker.seed(42)

RAW_DIR = '../../data/raw/invoice_cohorts'

# Load production invoice history
invoices_df = pd.read_csv(f'{RAW_DIR}/person_invoice_history_2025-04-20.csv')
invoices_df['createdat'] = pd.to_datetime(invoices_df['createdat'], utc=True, errors='coerce')

# Filter to invoices with a valid personid (some invoices have no linked person)
invoices_with_person = invoices_df[invoices_df['personid'].notna()].copy()

# Extract unique person IDs
person_ids = invoices_with_person['personid'].unique().tolist()

# Compute earliest invoice createdat per person (used as entry_date baseline in Step 2)
earliest_invoice_per_person = (
    invoices_with_person
    .groupby('personid')['createdat']
    .min()
    .reset_index()
    .rename(columns={'createdat': 'earliest_invoice_at'})
)

print(f"Production invoices loaded:     {len(invoices_df):,}")
print(f"Invoices with valid personid:   {len(invoices_with_person):,}")
print(f"Unique person IDs extracted:    {len(person_ids):,}")
print(f"\nInvoice date range:")
print(f"   Earliest: {invoices_with_person['createdat'].min()}")
print(f"   Latest:   {invoices_with_person['createdat'].max()}")
print(f"\nInvoice status distribution:")
status_labels = {4: 'PAID', 5: 'PARTIALLY_PAID', 6: 'UNPAID', 7: 'SCHEDULED', 8: 'CANCELED', 9: 'PENDING'}
for val, count in invoices_with_person['status'].value_counts().sort_index().items():
    label = status_labels.get(val, 'UNKNOWN')
    print(f"   {label} ({val}): {count} ({count / len(invoices_with_person) * 100:.1f}%)")
print(f"\nPerson ID sample (with earliest invoice date):")
earliest_invoice_per_person.head(5)

## Step 2: Generate Dummy Persons
Generates one synthetic `public.person` record per real production person ID.

**`entry_date` variation** (simulates different registration timelines):
- ~60% of persons: `entry_date` = earliest invoice `createdat` (person first appeared when invoiced)
- ~40% of persons: `entry_date` = earliest invoice `createdat` minus 30–365 random days (pre-existing patient with history before their first invoice in this dataset)

All other fields (`first_name`, `last_name`, `is_guardian`, etc.) are synthetic.

In [ ]:
person_records = []
entry_date_exact_count = 0
entry_date_backdated_count = 0

# Drop persons whose earliest invoice date could not be parsed (NaT)
earliest_invoice_per_person = earliest_invoice_per_person.dropna(subset=['earliest_invoice_at'])

for _, row in earliest_invoice_per_person.iterrows():
    person_id = row['personid']
    earliest_invoice = row['earliest_invoice_at']  # pd.Timestamp (UTC)

    # entry_date variation:
    # ~60%: exact earliest invoice createdat
    # ~40%: backdated 30–365 days before earliest invoice
    if np.random.rand() < 0.60:
        entry_date = earliest_invoice
        entry_date_exact_count += 1
    else:
        days_before = int(np.random.randint(30, 366))
        entry_date = earliest_invoice - timedelta(days=days_before)
        entry_date_backdated_count += 1

    modified_at = fake.date_time_between(
        start_date=entry_date.to_pydatetime(),
        end_date='now',
        tzinfo=pytz.UTC
    )

    person_records.append({
        'id': person_id,
        'org_id': str(uuid.uuid4()),
        'dataset_id': str(uuid.uuid4()),
        'source_tenant_id': str(uuid.uuid4()),
        'source_id': str(uuid.uuid4()),
        'person_pmid': fake.bothify(text='PM-######'),
        'household_id': str(uuid.uuid4()),
        'household_pmid': fake.bothify(text='HH-######'),
        'is_guardian': bool(np.random.choice([True, False], p=[0.25, 0.75])),
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'preferred_name': fake.first_name() if np.random.rand() < 0.3 else None,
        'status': np.random.choice(['Active', 'Inactive', 'Prospect'], p=[0.8, 0.1, 0.1]),
        'gender': np.random.choice(['Male', 'Female', 'Other', None], p=[0.45, 0.45, 0.05, 0.05]),
        'birthdate': fake.date_of_birth(minimum_age=18, maximum_age=85).isoformat(),
        'entry_date': entry_date.isoformat(),
        'created_at': entry_date.isoformat(),
        'modified_at': modified_at.isoformat(),
    })

persons_df = pd.DataFrame(person_records)

print(f"✅ Generated {len(persons_df)} dummy person records")
print(f"   Guardians: {persons_df['is_guardian'].sum()} ({persons_df['is_guardian'].mean() * 100:.1f}%)")
print(f"   Status distribution: {persons_df['status'].value_counts().to_dict()}")
print(f"   entry_date = earliest invoice: {entry_date_exact_count} ({entry_date_exact_count / len(persons_df) * 100:.1f}%)")
print(f"   entry_date backdated earlier:  {entry_date_backdated_count} ({entry_date_backdated_count / len(persons_df) * 100:.1f}%)")
print(f"\nPreview:")
persons_df[['id', 'first_name', 'last_name', 'is_guardian', 'entry_date']].head()

## Step 3: Generate Dummy Addresses
Generates one `public.address` record per person. Postal codes are sampled from the census dataset
(filtered to `census_match == 1` with complete valid data) to guarantee the census join in
`build_training_data_prod.ipynb` always succeeds with no nulls.

In [ ]:
import os

census_path = '../../data/processed/census_zip_features_model_ready.csv'
census_df = pd.read_csv(census_path, dtype={'zip_code': str})

# Only use zip codes with complete, valid census data
valid_census = census_df[
    (census_df['census_match'] == 1) &
    (census_df['median_household_income'].notna()) &
    (census_df['poverty_rate_pct'].notna()) &
    (census_df['poverty_rate_pct'] >= 0) &
    (census_df['unemployment_rate_pct'] >= 0) &
    (census_df['average_household_size'].notna()) &
    (census_df['average_household_size'] > 0)
].copy()

valid_zip_codes = valid_census['zip_code'].tolist()

print(f"Census zip codes total:             {len(census_df)}")
print(f"Valid zip codes with complete data: {len(valid_zip_codes)}")

address_records = []
for _, person in persons_df.iterrows():
    zip_code = np.random.choice(valid_zip_codes)
    address_records.append({
        'id': str(uuid.uuid4()),
        'org_id': person['org_id'],
        'dataset_id': person['dataset_id'],
        'source_tenant_id': person['source_tenant_id'],
        'source_id': str(uuid.uuid4()),
        'person_id': person['id'],
        'address_lines': fake.street_address(),
        'city': fake.city(),
        'state': fake.state_abbr(),
        'postal_code': zip_code,
        'created_at': person['created_at'],
        'modified_at': person['modified_at'],
    })

addresses_df = pd.DataFrame(address_records)

print(f"\n✅ Generated {len(addresses_df)} address records (1 per person)")
print(f"   Unique zip codes used: {addresses_df['postal_code'].nunique()}")
print(f"\nPreview:")
addresses_df[['person_id', 'city', 'state', 'postal_code']].head()

## Step 4: Generate Dummy Calendar Events
Generates `scheduler.calendars_events` records for each person. Each person receives a
`reliability_tendency` (sampled from Beta(5,2)) that weights their appointment status
distribution — mirroring the real-world pattern where some patients reliably attend while
others frequently cancel or no-show.

**Status distribution by reliability tier:**

| Tier | COMPLETED | CONFIRMED | CANCELED | NO_SHOW | UNCONFIRMED |
|---|---|---|---|---|---|
| High (>0.7) | 75% | 5% | 8% | 2% | 10% |
| Moderate (0.4–0.7) | 55% | 8% | 15% | 10% | 12% |
| Low (≤0.4) | 35% | 10% | 20% | 20% | 15% |

In [ ]:
ATTENDEE_STATUSES = ['CONFIRMED', 'COMPLETED', 'CANCELED', 'NO_SHOW', 'UNCONFIRMED']

calendar_event_records = []

location_id = str(uuid.uuid4())
source_tenant_id_cal = str(uuid.uuid4())
calendar_id = str(uuid.uuid4())

for _, person in persons_df.iterrows():
    person_entry = pd.Timestamp(person['entry_date'])

    # Number of appointments: 2–40, gamma-distributed (shape=4, scale=3 → mean ~12)
    num_appointments = max(2, int(np.random.gamma(shape=4, scale=3)))
    num_appointments = min(num_appointments, 40)

    # Person-level reliability tendency: Beta(5, 2) → skewed toward reliable
    reliability_tendency = float(np.random.beta(5, 2))

    for _ in range(num_appointments):
        apt_date = fake.date_time_between(
            start_date=person_entry.to_pydatetime(),
            end_date='now',
            tzinfo=pytz.UTC
        )

        if reliability_tendency > 0.7:
            status_probs = [0.05, 0.75, 0.08, 0.02, 0.10]  # mostly completed
        elif reliability_tendency > 0.4:
            status_probs = [0.08, 0.55, 0.15, 0.10, 0.12]  # moderate
        else:
            status_probs = [0.10, 0.35, 0.20, 0.20, 0.15]  # unreliable

        attendee_status = np.random.choice(ATTENDEE_STATUSES, p=status_probs)

        duration_minutes = int(np.random.choice([30, 45, 60, 90], p=[0.3, 0.35, 0.25, 0.1]))
        end_time = apt_date + timedelta(minutes=duration_minutes)

        calendar_event_records.append({
            'id': str(uuid.uuid4()),
            'location_id': location_id,
            'source_tenant_id': source_tenant_id_cal,
            'organizer_calendar_id': calendar_id,
            'organizer_id': str(uuid.uuid4()),
            'location_type': np.random.choice(['operatory', 'room'], p=[0.8, 0.2]),
            'type': 'APPOINTMENT',
            'attendee_id': person['id'],
            'attendee_status': attendee_status,
            'title': np.random.choice(['Cleaning', 'Check-up', 'Filling', 'Crown', 'Consultation', 'X-Ray']),
            'start_date': apt_date.date().isoformat(),
            'start_time': apt_date.isoformat(),
            'end_date': end_time.date().isoformat(),
            'end_time': end_time.isoformat(),
            'reference_id': str(uuid.uuid4()),
            'is_integrated': False,
            'recurring': False,
            'created_at': apt_date.isoformat(),
        })

calendar_events_df = pd.DataFrame(calendar_event_records)

print(f"✅ Generated {len(calendar_events_df)} calendar event records")
print(f"   Unique attendees: {calendar_events_df['attendee_id'].nunique()}")
print(f"   Avg appointments per person: {len(calendar_events_df) / calendar_events_df['attendee_id'].nunique():.1f}")
print(f"\nAttendee status distribution:")
for status, count in calendar_events_df['attendee_status'].value_counts().items():
    print(f"   {status}: {count} ({count / len(calendar_events_df) * 100:.1f}%)")
print(f"\nPreview:")
calendar_events_df[['attendee_id', 'attendee_status', 'title', 'start_date']].head()

## Step 5: Save Outputs

| File | Description |
|---|---|
| `prod_dummy_persons.csv` | Synthetic `public.person` records keyed to real production person IDs |
| `prod_dummy_addresses.csv` | Synthetic `public.address` records with real census zip codes |
| `prod_dummy_calendar_events.csv` | Synthetic `scheduler.calendars_events` with reliability-weighted statuses |

In [ ]:
output_dir = '../../data/processed'
os.makedirs(output_dir, exist_ok=True)

persons_output = os.path.join(output_dir, 'prod_dummy_persons.csv')
persons_df.to_csv(persons_output, index=False)
print(f"Saved prod_dummy_persons.csv:         {persons_output} ({len(persons_df)} rows)")

addresses_output = os.path.join(output_dir, 'prod_dummy_addresses.csv')
addresses_df.to_csv(addresses_output, index=False)
print(f"Saved prod_dummy_addresses.csv:       {addresses_output} ({len(addresses_df)} rows)")

calendar_events_output = os.path.join(output_dir, 'prod_dummy_calendar_events.csv')
calendar_events_df.to_csv(calendar_events_output, index=False)
print(f"Saved prod_dummy_calendar_events.csv: {calendar_events_output} ({len(calendar_events_df)} rows)")

print(f"\n✅ All dummy data saved to {os.path.abspath(output_dir)}")
print(f"\nSummary:")
print(f"   {len(persons_df)} persons (real IDs, synthetic attributes)")
print(f"   {len(addresses_df)} addresses (1 per person, real census zip codes)")
print(f"   {len(calendar_events_df)} calendar events ({len(calendar_events_df) / len(persons_df):.1f} avg per person)")
print(f"\nNext: run build_training_data_prod.ipynb to assemble the training table.")